<a href="https://colab.research.google.com/github/eternitvy/ML_26/blob/main/JS06/JS06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

df = pd.read_csv('voice.csv')

# Memisahkan fitur (X) dan label/target (y)
X = df.drop('label', axis=1)
y = df['label']

# **SOAL 1**

In [2]:
# ==========================================
# SOAL 1.a - Split Data
# ==========================================

# 1. Split dengan rasio 70:30 (test_size = 0.3)
X_train_70, X_test_30, y_train_70, y_test_30 = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Scaling fitur untuk rasio 70:30
scaler_70 = StandardScaler()
X_train_70_scaled = scaler_70.fit_transform(X_train_70)
X_test_30_scaled = scaler_70.transform(X_test_30)

# 2. Split dengan rasio 80:20 (test_size = 0.2)
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling fitur untuk rasio 80:20
scaler_80 = StandardScaler()
X_train_80_scaled = scaler_80.fit_transform(X_train_80)
X_test_20_scaled = scaler_80.transform(X_test_20)

In [3]:
# ==========================================
# SOAL 1.a.1 - Kernel Linear
# ==========================================

# Model Kernel Linear pada Rasio 70:30
model_linear_70 = SVC(kernel='linear')
model_linear_70.fit(X_train_70_scaled, y_train_70)
pred_linear_70 = model_linear_70.predict(X_test_30_scaled)
acc_linear_70 = accuracy_score(y_test_30, pred_linear_70)

# Model Kernel Linear pada Rasio 80:20
model_linear_80 = SVC(kernel='linear')
model_linear_80.fit(X_train_80_scaled, y_train_80)
pred_linear_80 = model_linear_80.predict(X_test_20_scaled)
acc_linear_80 = accuracy_score(y_test_20, pred_linear_80)

In [4]:
# ==========================================
# SOAL 1.a.2 - Kernel Polynomial
# ==========================================

# Model Kernel Poly pada Rasio 70:30
model_poly_70 = SVC(kernel='poly')
model_poly_70.fit(X_train_70_scaled, y_train_70)
pred_poly_70 = model_poly_70.predict(X_test_30_scaled)
acc_poly_70 = accuracy_score(y_test_30, pred_poly_70)

# Model Kernel Poly pada Rasio 80:20
model_poly_80 = SVC(kernel='poly')
model_poly_80.fit(X_train_80_scaled, y_train_80)
pred_poly_80 = model_poly_80.predict(X_test_20_scaled)
acc_poly_80 = accuracy_score(y_test_20, pred_poly_80)

In [5]:
# ==========================================
# SOAL 1.a.3 - Kernel RBF
# ==========================================

# Model Kernel RBF pada Rasio 70:30
model_rbf_70 = SVC(kernel='rbf')
model_rbf_70.fit(X_train_70_scaled, y_train_70)
pred_rbf_70 = model_rbf_70.predict(X_test_30_scaled)
acc_rbf_70 = accuracy_score(y_test_30, pred_rbf_70)

# Model Kernel RBF pada Rasio 80:20
model_rbf_80 = SVC(kernel='rbf')
model_rbf_80.fit(X_train_80_scaled, y_train_80)
pred_rbf_80 = model_rbf_80.predict(X_test_20_scaled)
acc_rbf_80 = accuracy_score(y_test_20, pred_rbf_80)

In [6]:
# ==========================================
# SOAL 1.b - Tabulasi Performansi Akurasi
# ==========================================

# Membuat ringkasan hasil akurasi dalam bentuk DataFrame
data_hasil = {
    'Rasio Split': ['70:30', '70:30', '70:30', '80:20', '80:20', '80:20'],
    'Kernel': ['Linear', 'Polynomial', 'RBF', 'Linear', 'Polynomial', 'RBF'],
    'Akurasi (%)': [
        round(acc_linear_70 * 100, 2),
        round(acc_poly_70 * 100, 2),
        round(acc_rbf_70 * 100, 2),
        round(acc_linear_80 * 100, 2),
        round(acc_poly_80 * 100, 2),
        round(acc_rbf_80 * 100, 2)
    ]
}

df_tabulasi = pd.DataFrame(data_hasil)
print(df_tabulasi)

  Rasio Split      Kernel  Akurasi (%)
0       70:30      Linear        97.06
1       70:30  Polynomial        95.79
2       70:30         RBF        98.11
3       80:20      Linear        97.63
4       80:20  Polynomial        97.16
5       80:20         RBF        98.26


# **SOAL 2**

In [10]:
import os
import cv2
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ==========================================
# 1. LOAD DATASET DARI GOOGLE DRIVE
# ==========================================
dataset_path = '/content/drive/MyDrive/images'

def extract_color_histogram(image, bins=(8, 8, 8)):
    """
    Ekstraksi histogram warna 3D dalam ruang warna RGB/HSV.
    """
    hist = cv2.calcHist([image], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

def load_data(root_dir, bins=(8, 8, 8)):
    data = []
    labels = []
    categories = ['day', 'night']

    # Memastikan folder dataset ditemukan
    if not os.path.exists(root_dir):
        raise FileNotFoundError(f"Folder '{root_dir}' tidak ditemukan. Pastikan Google Drive sudah di-mount.")

    for root, dirs, files in os.walk(root_dir):
        # Mengabaikan folder sisa metadata macOS jika ada
        if '__MACOSX' in root:
            continue

        category = os.path.basename(root)
        if category in categories:
            label = 0 if category == 'day' else 1
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')) and not file.startswith('._'):
                    img_path = os.path.join(root, file)
                    img = cv2.imread(img_path)

                    if img is not None:
                        features = extract_color_histogram(img, bins=bins)
                        data.append(features)
                        labels.append(label)

    return np.array(data), np.array(labels)

print("Memuat gambar dari Google Drive dan mengekstraksi fitur histogram...")
X, y = load_data(dataset_path, bins=(8, 8, 8))
print(f"Total data berhasil dimuat: {len(X)} sampel (Fitur: {X.shape[1]})")

# ==========================================
# 2. PEMBAGIAN DATASET (80:20)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Jumlah Data Latih (80%): {len(X_train)}")
print(f"Jumlah Data Uji   (20%): {len(X_test)}")

# ==========================================
# 3. EXPERIMENT: HYPERPARAMETER TUNING SVM (RBF KERNEL)
# ==========================================
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

print("\nMelakukan Hyperparameter Tuning menggunakan GridSearchCV...")
grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf'),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_cv_score = grid_search.best_score_
print(f"\nHyperparameter RBF Terbaik: {best_params}")
print(f"Akurasi Rata-rata Cross-Validation Latih: {best_cv_score * 100:.2f}%")

# ==========================================
# 4. EVALUASI MODEL PADA DATA UJI (20%)
# ==========================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*50)
print("PERFORMANSI EVALUASI MODEL SVM (RBF)")
print("="*50)
print(f"Akurasi Data Uji (Test Accuracy) : {accuracy * 100:.2f}%\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Day (Siang)', 'Night (Malam)']))


Memuat gambar dari Google Drive dan mengekstraksi fitur histogram...
Total data berhasil dimuat: 400 sampel (Fitur: 512)
Jumlah Data Latih (80%): 320
Jumlah Data Uji   (20%): 80

Melakukan Hyperparameter Tuning menggunakan GridSearchCV...

Hyperparameter RBF Terbaik: {'C': 1, 'gamma': 'scale'}
Akurasi Rata-rata Cross-Validation Latih: 99.38%

PERFORMANSI EVALUASI MODEL SVM (RBF)
Akurasi Data Uji (Test Accuracy) : 100.00%

Confusion Matrix:
[[40  0]
 [ 0 40]]

Classification Report:
               precision    recall  f1-score   support

  Day (Siang)       1.00      1.00      1.00        40
Night (Malam)       1.00      1.00      1.00        40

     accuracy                           1.00        80
    macro avg       1.00      1.00      1.00        80
 weighted avg       1.00      1.00      1.00        80

